In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import warnings
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.vector_ar.vecm import coint_johansen

warnings.filterwarnings('ignore')

In [2]:
pairs = pd.read_csv("./data/ah_pairs.csv")
pairs["A_Share"] = pairs["A_Share"].str.replace(".SH", ".SS", regex=False)
pairs["H_Share"] = pairs["H_Share"].str[1:]

pairs.head()

,Name,H_Share,Price_HKD,A_Share,Price_CNH,Premium (%)
0,FIRST TRACTOR,0038.HK,8.70,601038.SS,14.18,-44.47%
1,SICHUAN EXPRESS,0107.HK,5.42,601107.SS,6.26,-21.64%
2,TSINGTAO BREW,0168.HK,50.45,600600.SS,62.10,-26.48%
3,JIANGSU EXPRESS,0177.HK,9.93,600377.SS,12.00,-25.11%
4,JINGCHENG MAC,0187.HK,4.04,600860.SS,11.67,-68.67%


In [3]:
def load_data(ticker, start="2019-01-01"):
    df = yf.download(ticker, start=start, progress=False)
    return df[["Close", "Volume"]]

In [4]:
USD_CNY = yf.download("USDCNY=X", start="2016-01-01", progress=False)
USD_CNY = USD_CNY["Close"].iloc[:, 0]   # force Series

USD_HKD = yf.download("USDHKD=X", start="2016-01-01", progress=False)
USD_HKD = USD_HKD["Close"].iloc[:, 0]

FX = USD_CNY / USD_HKD
FX.dropna().head()

YF.download() has changed argument auto_adjust default to True


Date
2016-01-01    0.836595
2016-01-04    0.836566
2016-01-05    0.841850
2016-01-06    0.840202
2016-01-07    0.844500
dtype: float64

In [5]:
results = []

for _, row in pairs.iterrows():
    try:
        A = load_data(row["A_Share"])
        H = load_data(row["H_Share"])

        df = pd.concat([A, H], axis=1, keys=["A", "H"]).dropna()
        df = df.sort_index()
        df = df.ffill().dropna()
        df.columns = [ f"{lvl0}_{lvl1}" for lvl0, lvl1, _ in df.columns ]
        df["H_CNY"] = df["H_Close"] * FX.reindex(df.index).ffill().bfill()
        df["premium"] = (df["A_Close"] - df["H_CNY"]) / df["H_CNY"]

        # Liquidity
        adv_A = (df["A_Close"] * df["A_Volume"]).mean()
        adv_H = (df["H_Close"] * df["H_Volume"]).mean()

        # Premium stats
        prem_mean = df["premium"].mean()
        prem_std = df["premium"].std()

        # Mean reversion proxy (half-life)
        dp = df["premium"].diff().dropna()
        lag = df["premium"].shift(1).dropna()
        beta = np.polyfit(lag.loc[dp.index], dp, 1)[0]
        half_life = -np.log(2) / beta if beta < 0 else np.inf

        results.append({
            "company": row["Name"],
            "A": row["A_Share"],
            "H": row["H_Share"],
            "ADV_A": adv_A,
            "ADV_H": adv_H,
            "prem_mean": prem_mean,
            "prem_std": prem_std,
            "half_life": half_life
        })

    except Exception as e:
        print("FAILED FOR:", row["Name"])
        print("A:", row["A_Share"], "H:", row["H_Share"])
        raise

rank_df = pd.DataFrame(results)

In [6]:
rank_df_dummy = rank_df

In [7]:
rank_df2 = rank_df[
    (rank_df["ADV_A"] > 5e7) &      
    (rank_df["ADV_H"] > 5e7) &
    (rank_df["half_life"] < 20)    # mean reversion < ~1 month
]

rank_df2["score"] = (
    rank_df2["prem_std"].rank(pct=True) * 0.5 +
    (-rank_df2["half_life"]).rank(pct=True) * 0.5
)

rank_df2 = rank_df2.sort_values("score", ascending=False)
print(f"Number of companies with arbitrage possibility = {rank_df2.shape[0]}")
rank_df2.head(10)

Number of companies with arbitrage possibility = 3


,company,A,H,ADV_A,ADV_H,prem_mean,prem_std,half_life,score
5,MIDEA GROUP,000333.SZ,0300.HK,2.538756e+09,3.778222e+08,0.120439,1.551523,0.694269,1.0
115,CATL,300750.SZ,3750.HK,9.279281e+09,1.313144e+09,-0.196859,0.059392,16.961744,0.5
137,HAIER SMARTHOME,600690.SS,6690.HK,8.689780e+08,3.111103e+08,0.132106,0.044779,8.135767,0.5


In [8]:
def engle_granger_test(A, H, min_obs=252):
    """
    A, H : pd.Series (prices, already aligned, no NaNs)
    Returns dict with beta, adf_stat, pvalue, half_life
    """

    if len(A) < min_obs or len(H) < min_obs:
        return None

    X = sm.add_constant(H.values)
    model = sm.OLS(A.values, X).fit()
    beta = model.params[1]

    resid = model.resid
    adf_stat, pvalue, _, _, _, _ = adfuller(resid, autolag="AIC")

    resid_series = pd.Series(resid)
    d_resid = resid_series.diff().dropna()
    lag_resid = resid_series.shift(1).dropna()

    if len(d_resid) < 50:
        half_life = np.inf
    else:
        kappa = np.polyfit(lag_resid.loc[d_resid.index], d_resid, 1)[0]
        half_life = -np.log(2) / kappa if kappa < 0 else np.inf

    return {
        "beta": beta,
        "adf_stat": adf_stat,
        "pvalue": pvalue,
        "half_life": half_life
    }


In [9]:
eg_results = []

for _, row in rank_df2.iterrows():
    try:
        A = load_data(row["A"])
        H = load_data(row["H"])

        if A is None or H is None:
            continue

        df = pd.concat([A, H], axis=1, keys=["A", "H"])
        df = df.sort_index().ffill().dropna()
        df.columns = [f"{a}_{b}" for a, b, _ in df.columns]

        A_px = df["A_Close"]
        H_px = df["H_Close"]

        res = engle_granger_test(A_px, H_px)

        if res is None:
            continue

        eg_results.append({
            "company": row["company"],
            "A": row["A"],
            "H": row["H"],
            **res
        })

    except Exception as e:
        print("FAILED:", row["company"], e)


In [10]:
eg_df = pd.DataFrame(eg_results)

# cointegration filter
eg_df = eg_df[(eg_df["pvalue"] < 0.05) & (eg_df["beta"] > 0)]

In [11]:
print(f"Number of stocks after filtering via EG test = {eg_df.shape[0]}")
eg_df = eg_df.sort_values("pvalue")
eg_df.head(10)

Number of stocks after filtering via EG test = 0


,company,A,H,beta,adf_stat,pvalue,half_life


In [12]:
def johansen_test(A, H, det_order=0, k_ar_diff=1):
    """
    A, H : pd.Series (aligned, no NaNs)
    Returns cointegration rank (0, 1, or 2)
    """

    data = np.column_stack([A.values, H.values])

    result = coint_johansen(data, det_order, k_ar_diff)

    trace_stat = result.lr1
    crit_vals = result.cvt[:, 1]   

    rank = 0
    for stat, crit in zip(trace_stat, crit_vals):
        if stat > crit:
            rank += 1

    return rank

In [13]:
johansen_results = []

for _, row in eg_df.iterrows():
    try:
        A = load_data(row["A"])
        H = load_data(row["H"])

        if A is None or H is None:
            continue

        df = pd.concat([A, H], axis=1, keys=["A", "H"])
        df = df.sort_index().ffill().dropna()

        # flatten columns
        df.columns = [f"{a}_{b}" for a, b, _ in df.columns]

        A_px = df["A_Close"]
        H_px = df["H_Close"]

        if len(df) < 252:
            continue

        rank = johansen_test(A_px, H_px)

        johansen_results.append({
            "company": row["company"],
            "A": row["A"],
            "H": row["H"],
            "johansen_rank": rank
        })

    except Exception as e:
        print("FAILED:", row["company"], e)


In [14]:
johansen_df = pd.DataFrame(johansen_results)

final_df = eg_df.merge(
    johansen_df,
    on=["company", "A", "H"],
    how="inner"
)

final_df.head()

KeyError: 'company'

In [ ]:
final_df = final_df[final_df["johansen_rank"] == 1]
print(f"Number of stocks left after Johansen test = {final_df.shape[0]}")
final_df.head()

In [ ]:
rank_df2[["company", "A", "H"]].to_csv("./data/ah_pairs_arb2.csv", index=False)